# Ising and Potts updates in Julia

I build one local Monte Carlo update before repeating it. The square lattice is periodic. In the Ising model a spin has value -1 or +1; the dimensionless interaction strength is one and temperature is measured in units of the interaction energy divided by Boltzmann’s constant. For Potts grain growth the integers identify grains, not compositions. Neighbor-copy proposals describe the supplied grain-growth kinetics and should not be presented as unbiased equilibrium sampling.

Run each small part and its checks in order. Julia indices start at 1. For Fourier examples install FFTW with `julia -e 'using Pkg; Pkg.add("FFTW")'`. For notebook use, install IJulia and select its kernel. The script is also supplied. Julia and NumPy random sequences differ; compare ensemble statistics, not individual paths.

**Using my code.** I permit free noncommercial teaching, learning and demonstrations with acknowledgment. Research (including academic research) and commercial use require my explicit written permission. Earlier MIT and GPL releases retain their existing permissions. Read the [code-use terms](https://saswataiith.github.io/files/CODE-USE-TERMS.txt). Request permission at saswata_at_msme.iith.ac.in. These terms apply to code; teaching text and figures have separate terms.


## 1. Inspect periodic neighbors

I first test the boundary indexing on a grid of known values. Opposite edges are adjacent. Rows and columns identify sites; they are dimensionless indices.

In [ ]:
using Random
rng = MersenneTwister(7)
function neighbor_values(state, row, column)
    rows, columns = size(state)
    return [state[mod1(row+1,rows),column], state[mod1(row-1,rows),column],
            state[row,mod1(column+1,columns)], state[row,mod1(column-1,columns)]]
end
test_grid = [0 1 2; 3 4 5; 6 7 8]
@assert neighbor_values(test_grid, 1, 1) == [3, 6, 1, 2]
println("Periodic neighbor indexing passed.")

## 2. Calculate the Ising energy change

Flipping a spin changes each of its four bond energies. In a uniform positive-spin neighborhood the change is +8. I check this before using a probability rule.

In [ ]:
function ising_energy_change(old_spin, neighbors)
    return 2 * old_spin * sum(neighbors)
end
@assert ising_energy_change(1, [1,1,1,1]) == 8
@assert ising_energy_change(-1, [1,1,1,1]) == -8
println("Ising energy changes passed.")

## 3. Accept or reject a trial

I always accept a decrease in energy. For an increase I accept with probability exp(-ΔE/T). The dimensionless temperature T must be positive in this implementation. This rule permits occasional uphill moves; energy need not decrease at every update.

In [ ]:
function accept_trial(energy_change, temperature, rng)
    temperature > 0 || error("Use positive temperature in this example.")
    if energy_change <= 0
        return true
    end
    return rand(rng) < exp(-energy_change / temperature)
end
@assert accept_trial(-8, 1.5, rng)
@assert accept_trial(0, 1.5, rng)
@assert 0 < exp(-8 / 1.5) < 1
println("Acceptance checks passed.")

## 4. Make and repeat one Ising trial

I choose a random site with replacement. One sweep consists of as many attempted updates as there are sites, so some sites can be chosen more than once. A spin flip does not conserve total spin.

In [ ]:
function ising_trial!(state, temperature, rng)
    row = rand(rng, axes(state,1))
    column = rand(rng, axes(state,2))
    old = state[row,column]
    change = ising_energy_change(old, neighbor_values(state,row,column))
    if accept_trial(change, temperature, rng)
        state[row,column] = -old
    end
end
spins = rand(rng, [-1,1], 16,16)
for trial in 1:length(spins)*10
    ising_trial!(spins, 1.5, rng)
end
@assert all(value in (-1,1) for value in spins)
println("Ising state values: ", unique(spins))

## 5. Change the local rule for Potts grain growth

For a Potts trial I propose a neighbor’s label and count unlike neighbors before and after the change. The energy change is the difference in these counts, in units of the grain-boundary interaction. No new grain label can be created by copying a neighbor. I check both the local change and that property.

In [ ]:
function potts_energy_change(old, candidate, neighbors)
    return count(!=(candidate), neighbors) - count(!=(old), neighbors)
end
@assert potts_energy_change(1,2,[2,2,2,1]) == -2
function potts_trial!(state, temperature, rng)
    row = rand(rng, axes(state,1))
    column = rand(rng, axes(state,2))
    neighbors = neighbor_values(state,row,column)
    candidate = rand(rng, neighbors)
    change = potts_energy_change(state[row,column],candidate,neighbors)
    if accept_trial(change,temperature,rng)
        state[row,column] = candidate
    end
end
grains = reshape(randperm(rng,256),16,16)
initial_labels = Set(grains)
for trial in 1:length(grains)*10
    potts_trial!(grains,0.3,rng)
end
@assert issubset(Set(grains),initial_labels)
println("Grains initially: ", length(initial_labels), "; remaining labels: ", length(unique(grains)))